# Lost-words / Zgubione słowa — Short Video Pipeline

Automatyczny pipeline do tworzenia pionowych odcinków survival / found-footage: scenariusz JSON → lektor → kadry → napisy → MP4 1080×1920.

**Domyślnie bezpłatny tryb startowy:** szablon scenariusza, `edge-tts` i syntetyczne kadry testowe. Klucz API jest potrzebny tylko po włączeniu wybranego adaptera LLM / generatora obrazów. Nie wklejaj kluczy bezpośrednio do komórek notatnika.

Uruchamiaj komórki po kolei. Notebook montuje Google Drive w `/content/drive/MyDrive/lost_words_project`.

In [ ]:
# KOMÓRKA 1/6 — środowisko, Google Drive, ffmpeg i zależności
from pathlib import Path
import os, sys, json, re, math, time, random, shutil, hashlib, subprocess, textwrap
from datetime import datetime

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    raise RuntimeError(
        "Nie udało się zamontować Google Drive. Uruchom ten notebook w Google Colab "
        "i zaakceptuj dostęp do Dysku. Szczegóły: " + str(exc)
    )

PROJECT_ROOT = Path('/content/drive/MyDrive/lost_words_project')
ASSETS_DIR = PROJECT_ROOT / 'assets'
AUDIO_DIR = PROJECT_ROOT / 'audio'
FRAMES_DIR = PROJECT_ROOT / 'frames'
TEMP_DIR = PROJECT_ROOT / 'temp'
OUTPUT_DIR = PROJECT_ROOT / 'output'
for folder in (PROJECT_ROOT, ASSETS_DIR, AUDIO_DIR, FRAMES_DIR, TEMP_DIR, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def run_cmd(args, *, check=True, capture_output=True):
    """Uruchamia proces i pokazuje czytelny komunikat przy błędzie."""
    try:
        result = subprocess.run(
            [str(x) for x in args], check=check, text=True,
            stdout=subprocess.PIPE if capture_output else None,
            stderr=subprocess.PIPE if capture_output else None
        )
        return result
    except subprocess.CalledProcessError as exc:
        stderr = (exc.stderr or '')[-4000:]
        print(f"[BŁĄD] Polecenie zakończyło się kodem {exc.returncode}.")
        if stderr:
            print(stderr)
        raise
    except FileNotFoundError as exc:
        raise RuntimeError(f"Nie znaleziono programu: {args[0]}. Szczegóły: {exc}") from exc

# Instalacja narzędzi systemowych tylko wtedy, gdy brakuje ffmpeg.
try:
    if shutil.which('ffmpeg') is None or shutil.which('ffprobe') is None:
        print('[INFO] Instaluję ffmpeg i podstawowe fonty…')
        run_cmd(['apt-get', 'update', '-qq'], capture_output=False)
        run_cmd(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'fonts-dejavu-core'], capture_output=False)
    else:
        print('[OK] ffmpeg i ffprobe są dostępne.')
except Exception as exc:
    raise RuntimeError(f"Konfiguracja ffmpeg nie powiodła się: {exc}") from exc

try:
    run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'requests', 'Pillow', 'edge-tts'])
    import requests
    from PIL import Image, ImageDraw, ImageFilter, ImageEnhance, ImageOps
except Exception as exc:
    raise RuntimeError(f"Instalacja/import pakietów nie powiódł się: {exc}") from exc

# Kontrola wersji i filtrów wymaganych przez montaż.
try:
    print('[INFO] ffmpeg:', run_cmd(['ffmpeg', '-version']).stdout.splitlines()[0])
    filter_result = run_cmd(['ffmpeg', '-hide_banner', '-filters'])
    filters = (filter_result.stdout or '') + (filter_result.stderr or '')
    if ' subtitles ' not in filters and 'subtitles' not in filters:
        print('[OSTRZEŻENIE] Ten build ffmpeg może nie zawierać filtra subtitles; montaż zgłosi to ponownie przy eksporcie.')
    print(f'[OK] Katalog projektu: {PROJECT_ROOT}')
except Exception as exc:
    raise RuntimeError(f"Nie udało się zweryfikować ffmpeg: {exc}") from exc

In [ ]:
# KOMÓRKA 2/6 — konfiguracja odcinka i bezpieczne pobranie sekretów

def get_secret(name, default=None):
    """Najpierw Colab Secrets, potem zmienna środowiskowa. Nigdy nie wypisuje wartości."""
    value = None
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    if not value:
        value = os.environ.get(name)
    return value if value else default

SERIES_NAME = 'Lost-words / Zgubione słowa'
EPISODE_TOPIC = 'A strange radio transmission from an abandoned shelter'
EPISODE_LOCATION = 'Łódź, Poland'
STORY_LANGUAGE = 'PL'       # 'PL' albo 'EN'
NUM_SCENES = 6
TARGET_SECONDS = 40         # docelowo 35–45 s; faktyczny czas zależy od tempa lektora
VIDEO_WIDTH, VIDEO_HEIGHT, FPS = 1080, 1920, 30
VISUAL_STYLE = 'gritty analog found footage, desaturated horror, 35mm film grain, imperfect handheld documentary realism'

# Domyślne ustawienia nie uruchamiają płatnego generowania obrazów.
LLM_PROVIDER = 'openai_compatible'  # lub 'template'
IMAGE_PROVIDER = 'fallback'         # 'fallback' albo 'openai'
TTS_PROVIDER = 'edge'               # domyślnie darmowy edge-tts; adapter można podmienić
ALLOW_SILENT_TEST_AUDIO = True      # awaryjnie pozwala sprawdzić montaż bez działającego TTS
FALLBACK_ON_IMAGE_ERROR = True
IMAGE_RETRIES = 3

OPENAI_API_KEY = get_secret('OPENAI_API_KEY')
OPENAI_BASE_URL = (get_secret('OPENAI_BASE_URL', 'https://api.openai.com/v1') or '').rstrip('/')
OPENAI_CHAT_MODEL = get_secret('OPENAI_CHAT_MODEL', 'gpt-4o-mini')
OPENAI_IMAGE_MODEL = get_secret('OPENAI_IMAGE_MODEL', 'gpt-image-2.5-flare')
TTS_API_KEY = get_secret('TTS_API_KEY')  # używany wyłącznie po podłączeniu własnego adaptera TTS

VOICE_BY_LANGUAGE = {
    'PL': 'pl-PL-ZofiaNeural',
    'EN': 'en-US-JennyNeural',
}
if STORY_LANGUAGE not in VOICE_BY_LANGUAGE:
    raise ValueError("STORY_LANGUAGE musi mieć wartość 'PL' albo 'EN'.")
if not 4 <= NUM_SCENES <= 10:
    raise ValueError('NUM_SCENES ustaw w zakresie 4–10, aby utrzymać tempo shorta.')
if not 35 <= TARGET_SECONDS <= 45:
    raise ValueError('TARGET_SECONDS ustaw w zakresie 35–45 sekund.')

VOICE = VOICE_BY_LANGUAGE[STORY_LANGUAGE]
EPISODE_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
EPISODE_ID = re.sub(r'[^a-z0-9]+', '-', f"lost-words-{EPISODE_STAMP}".lower()).strip('-')
FRAME_EPISODE_DIR = FRAMES_DIR / EPISODE_ID
FRAME_EPISODE_DIR.mkdir(parents=True, exist_ok=True)

print(f'[OK] Seria: {SERIES_NAME} | język: {STORY_LANGUAGE} | sceny: {NUM_SCENES}')
print(f'[OK] Temat: {EPISODE_TOPIC} | lokalizacja: {EPISODE_LOCATION}')
print(f'[INFO] LLM: {LLM_PROVIDER} | obrazy: {IMAGE_PROVIDER} | TTS: {TTS_PROVIDER}')
print('[INFO] OPENAI_API_KEY:', 'dostępny' if OPENAI_API_KEY else 'brak — zostanie użyty tryb szablonowy / testowy')

In [ ]:
# KOMÓRKA 3/6 — scenariusz LLM: czysty JSON + bezpieczny fallback

def strip_json_fence(raw_text):
    """Usuwa markdownowe ```json ... ``` i wycina obiekt JSON z odpowiedzi."""
    if not isinstance(raw_text, str) or not raw_text.strip():
        raise ValueError('Model zwrócił pustą odpowiedź.')
    cleaned = re.sub(r'^\s*```(?:json)?\s*', '', raw_text.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r'\s*```\s*$', '', cleaned)
    start, end = cleaned.find('{'), cleaned.rfind('}')
    if start < 0 or end <= start:
        raise ValueError('W odpowiedzi modelu nie znaleziono kompletnego obiektu JSON.')
    return cleaned[start:end + 1]


def validate_story(payload):
    if not isinstance(payload, dict):
        raise TypeError('Scenariusz musi być obiektem JSON.')
    for required in ('title', 'hook', 'scenes'):
        if required not in payload:
            raise ValueError(f'Brak wymaganego pola: {required}')
    if not isinstance(payload['title'], str) or not payload['title'].strip():
        raise ValueError('Pole title musi być niepustym tekstem.')
    if not isinstance(payload['hook'], str) or not payload['hook'].strip():
        raise ValueError('Pole hook musi być niepustym tekstem.')
    scenes = payload['scenes']
    if not isinstance(scenes, list) or len(scenes) != NUM_SCENES:
        raise ValueError(f'Wymagane jest dokładnie {NUM_SCENES} scen.')
    normalized = []
    for idx, scene in enumerate(scenes, start=1):
        if not isinstance(scene, dict):
            raise TypeError(f'Scena {idx} nie jest obiektem JSON.')
        for field in ('scene_id', 'narration', 'visual_prompt'):
            if not isinstance(scene.get(field), str) or not scene[field].strip():
                raise ValueError(f'Scena {idx}: brak poprawnego pola {field}.')
        normalized.append({
            'scene_id': str(scene['scene_id']).strip(),
            'narration': str(scene['narration']).strip(),
            'visual_prompt': str(scene['visual_prompt']).strip(),
        })
    hook = payload['hook'].strip()
    # Hook powinien dać się wypowiedzieć w pierwszych ~3 sekundach.
    words = hook.split()
    if len(words) > 7:
        hook = ' '.join(words[:7]).rstrip(' ,;:') + '…'
        print('[OSTRZEŻENIE] Skrócono hook do 7 słów, aby zmieścić go na początku shorta.')
    return {'title': payload['title'].strip(), 'hook': hook, 'scenes': normalized}


def fallback_story():
    if STORY_LANGUAGE == 'PL':
        title = 'Sygnał, którego nie powinno być'
        hook = 'Jeśli to słyszysz, nie odpowiadaj.'
        narrations = [
            'Radio obudziło mnie dokładnie o trzeciej siedemnaście. Nikt nie nadawał od miesięcy.',
            'Wiadomość była krótka: schron jest pusty, ale światło nadal się pali.',
            'Wyszedłem na ulicę. Miasto wyglądało tak, jakby wszyscy zniknęli w połowie zdania.',
            'Na częstotliwości usłyszałem własny głos. Mówił, żebym nie szedł na północ.',
            'W dzienniku znalazłem wpis z jutrzejszą datą. Ktoś opisał dokładnie tę noc.',
            'Nadajnik znów zaszumiał. Tym razem głos powiedział moje imię i urwał połączenie.',
            'Zabrałem radio na dach, ale antena wskazywała pusty horyzont.',
            'W kieszeni znalazłem kartkę z jednym zdaniem: nie jesteś już sam.',
            'O świcie sygnał wrócił. Tym razem ktoś po drugiej stronie oddychał.',
            'Wyłączyłem odbiornik, lecz głos nadal brzmiał w słuchawkach.',
        ]
    else:
        title = 'The Signal That Should Not Exist'
        hook = 'If you hear this, do not answer.'
        narrations = [
            'The radio woke me at exactly seventeen past three. Nobody had transmitted for months.',
            'The message was short: the shelter is empty, but its lights are still on.',
            'I stepped outside. The city looked as if everyone had vanished in the middle of a sentence.',
            'On the frequency, I heard my own voice telling me not to go north.',
            'The journal contained an entry dated tomorrow. Someone had described this exact night.',
            'The transmitter crackled again. This time the voice said my name, then the signal died.',
            'I carried the radio to the roof, but the antenna faced an empty horizon.',
            'A note in my pocket had one sentence: you are not alone anymore.',
            'At sunrise the signal returned. Someone was breathing on the other side.',
            'I switched the receiver off, but the voice remained in my headphones.',
        ]
    visual_prompts = [
        'Abandoned radio room at night, dusty analogue receiver glowing faintly, rain on a cracked window, no people, vertical cinematic composition.',
        'Empty underground shelter corridor with one distant practical light, damp concrete, scattered paper, quiet tension, vertical framing.',
        'Deserted Polish city street before dawn, abandoned cars, drifting fog, damaged streetlights, distant indistinct shadow only.',
        'Close view of an old radio microphone and tangled cables, weak red indicator light, shallow depth of field, documentary realism.',
        'Open survival journal on a metal desk beside a flashlight, illegible marks only, dust in a narrow beam of light, no readable text.',
        'Lonely rooftop antenna against a pale overcast sky, wind moving loose cables, empty city skyline, ominous analog film atmosphere.',
        'Close portrait of a handheld radio on a wet rooftop ledge, distant empty skyline, cold dawn light, no visible text.',
        'A folded blank note beside a worn jacket pocket and flashlight, cinematic macro photography, dark muted palette.',
        'Rooftop at sunrise with a radio antenna and a distant tiny light in the fog, ambiguous hopeful unease.',
        'An empty pair of headphones beside a switched-off radio receiver in a dark shelter room, faint window light.',
    ]
    return validate_story({
        'title': title,
        'hook': hook,
        'scenes': [
            {'scene_id': f'S{idx:02d}', 'narration': narrations[idx-1], 'visual_prompt': visual_prompts[idx-1]}
            for idx in range(1, NUM_SCENES + 1)
        ]
    })


def generate_story_openai():
    if not OPENAI_API_KEY:
        raise RuntimeError('Brak OPENAI_API_KEY w Colab Secrets.')
    system_prompt = (
        'You are a cinematic short-form audio-drama writer. Return ONLY valid JSON, with no markdown. '
        'The story is fictional, first-person survival journal / radio transmission in a non-graphic zombie-apocalypse setting. '
        'Keep it suspenseful, atmospheric, plausible and safe; no gore, no graphic injuries, no instructions for real-world harm. '
        'Every visual_prompt MUST be in English and explicitly request no text, lettering, logos, captions or watermark in the image.'
    )
    user_prompt = f"""Create one {TARGET_SECONDS}-second vertical short for the series "{SERIES_NAME}".
Topic: {EPISODE_TOPIC}
Location: {EPISODE_LOCATION}
Narration language: {STORY_LANGUAGE}
Visual style: {VISUAL_STYLE}
Required scene count: exactly {NUM_SCENES}.

Return this exact JSON structure:
{{"title":"...","hook":"...","scenes":[{{"scene_id":"S01","narration":"...","visual_prompt":"..."}}]}}
Rules: hook is in {STORY_LANGUAGE}, 4–7 spoken words, immediately gripping and suitable for the first 3 seconds. Each narration is in {STORY_LANGUAGE}, first person, radio-log / recovered-diary tone, concise. Combined narration plus hook should target 80–100 spoken words. Give each scene a distinct visual_prompt in English, portrait 9:16 composition, grounded cinematic detail, no visible text or typography. Avoid gore.
"""
    endpoint = f'{OPENAI_BASE_URL}/chat/completions'
    response = requests.post(
        endpoint,
        headers={'Authorization': f'Bearer {OPENAI_API_KEY}', 'Content-Type': 'application/json'},
        json={
            'model': OPENAI_CHAT_MODEL,
            'response_format': {'type': 'json_object'},
            'messages': [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_prompt},
            ],
            'temperature': 0.8,
        },
        timeout=90,
    )
    response.raise_for_status()
    data = response.json()
    try:
        content = data['choices'][0]['message']['content']
    except (KeyError, IndexError, TypeError) as exc:
        raise ValueError('Nieoczekiwany format odpowiedzi endpointu chat/completions.') from exc
    return validate_story(json.loads(strip_json_fence(content)))


try:
    if LLM_PROVIDER == 'openai_compatible' and OPENAI_API_KEY:
        STORY = generate_story_openai()
        print('[OK] Scenariusz wygenerowany przez API.')
    else:
        STORY = fallback_story()
        print('[INFO] Użyto wbudowanego scenariusza testowego (bez klucza LLM).')
except Exception as exc:
    print(f'[OSTRZEŻENIE] LLM nie wygenerował poprawnego scenariusza: {type(exc).__name__}: {exc}')
    print('[INFO] Przechodzę na bezpieczny scenariusz testowy.')
    STORY = fallback_story()

story_path = ASSETS_DIR / f'{EPISODE_ID}_story.json'
story_path.write_text(json.dumps(STORY, ensure_ascii=False, indent=2), encoding='utf-8')
print('\nTYTUŁ:', STORY['title'])
print('HOOK:', STORY['hook'])
for scene in STORY['scenes']:
    print(f"{scene['scene_id']}: {scene['narration']}")
print(f'[OK] JSON zapisany: {story_path}')

In [ ]:
# KOMÓRKA 4/6 — TTS: edge-tts, osobne segmenty, fallback ciszy do testu montażu
import asyncio

SEGMENT_SPECS = [{'segment_id': 'hook', 'text': STORY['hook']}] + [
    {'segment_id': scene['scene_id'], 'text': scene['narration']}
    for scene in STORY['scenes']
]


def probe_duration(path):
    result = run_cmd([
        'ffprobe', '-v', 'error', '-show_entries', 'format=duration',
        '-of', 'default=noprint_wrappers=1:nokey=1', str(path)
    ])
    try:
        duration = float(result.stdout.strip())
        if not math.isfinite(duration) or duration <= 0:
            raise ValueError('Niepoprawny czas trwania.')
        return duration
    except Exception as exc:
        raise RuntimeError(f'Nie można odczytać długości pliku audio {path}: {exc}') from exc


async def create_edge_tts_segment(text, mp3_path):
    try:
        import edge_tts
        communicate = edge_tts.Communicate(text=text, voice=VOICE, rate='+0%', volume='+0%')
        await communicate.save(str(mp3_path))
    except Exception as exc:
        raise RuntimeError(f'edge-tts nie wygenerował segmentu: {type(exc).__name__}: {exc}') from exc


async def create_external_tts_segment(text, mp3_path):
    """Miejsce na własny TTS. Adapter ma zapisać audio segmentu do mp3_path."""
    if not TTS_API_KEY:
        raise RuntimeError('Własny TTS wymaga sekretu TTS_API_KEY.')
    raise NotImplementedError(
        'Wstaw tu wywołanie wybranego dostawcy TTS (requests.post), a odpowiedź audio zapisz do mp3_path.'
    )


def make_silent_test_segments(segment_specs):
    """Generuje ciszę o przewidywanych długościach — tylko do testowania montażu."""
    hook_duration = 2.6
    remaining = max(10.0, float(TARGET_SECONDS) - hook_duration)
    weights = [max(5, len(item['text'].split())) for item in segment_specs[1:]]
    total_weight = sum(weights) or 1
    durations = [hook_duration] + [remaining * w / total_weight for w in weights]
    silent_files, measured = [], []
    for idx, (item, duration) in enumerate(zip(segment_specs, durations)):
        wav_path = AUDIO_DIR / f'{EPISODE_ID}_{idx:02d}_{item["segment_id"]}_silent.wav'
        run_cmd([
            'ffmpeg', '-y', '-f', 'lavfi', '-t', f'{duration:.3f}',
            '-i', 'anullsrc=channel_layout=stereo:sample_rate=48000',
            '-c:a', 'pcm_s16le', str(wav_path)
        ])
        silent_files.append(wav_path)
        measured.append(probe_duration(wav_path))
    return silent_files, measured


AUDIO_SEGMENT_FILES, SEGMENT_DURATIONS = [], []
try:
    if TTS_PROVIDER not in {'edge', 'external'}:
        raise ValueError("TTS_PROVIDER musi mieć wartość 'edge' albo 'external'.")
    for idx, item in enumerate(SEGMENT_SPECS):
        mp3_path = AUDIO_DIR / f'{EPISODE_ID}_{idx:02d}_{item["segment_id"]}.mp3'
        wav_path = AUDIO_DIR / f'{EPISODE_ID}_{idx:02d}_{item["segment_id"]}.wav'
        if TTS_PROVIDER == 'edge':
            await create_edge_tts_segment(item['text'], mp3_path)
        else:
            await create_external_tts_segment(item['text'], mp3_path)
        if not mp3_path.exists() or mp3_path.stat().st_size < 1000:
            raise RuntimeError(f'Plik TTS jest pusty lub niekompletny: {mp3_path.name}')
        run_cmd(['ffmpeg', '-y', '-i', str(mp3_path), '-ar', '48000', '-ac', '2', '-c:a', 'pcm_s16le', str(wav_path)])
        AUDIO_SEGMENT_FILES.append(wav_path)
        SEGMENT_DURATIONS.append(probe_duration(wav_path))
    print('[OK] Lektor edge-tts wygenerowany dla wszystkich segmentów.')
except Exception as exc:
    print(f'[OSTRZEŻENIE] Synteza mowy nie powiodła się: {type(exc).__name__}: {exc}')
    if not ALLOW_SILENT_TEST_AUDIO:
        raise
    print('[INFO] Tworzę ścieżkę ciszy do sprawdzenia montażu. To nie jest gotowa narracja.')
    AUDIO_SEGMENT_FILES, SEGMENT_DURATIONS = make_silent_test_segments(SEGMENT_SPECS)

# Łączenie WAV bez ponownej kompresji; wszystkie segmenty mają 48 kHz stereo.
concat_audio_list = TEMP_DIR / f'{EPISODE_ID}_audio_concat.txt'
concat_audio_list.write_text('\n'.join(f"file '{p.as_posix()}'" for p in AUDIO_SEGMENT_FILES), encoding='utf-8')
FINAL_AUDIO = AUDIO_DIR / f'{EPISODE_ID}_narration.wav'
run_cmd(['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', str(concat_audio_list), '-c:a', 'pcm_s16le', str(FINAL_AUDIO)])
ACTUAL_AUDIO_SECONDS = probe_duration(FINAL_AUDIO)
print(f'[OK] Audio: {FINAL_AUDIO}')
print(f'[INFO] Długość audio: {ACTUAL_AUDIO_SECONDS:.2f} s | cel: {TARGET_SECONDS} s')

In [ ]:
# KOMÓRKA 5/6 — kadry 9:16: OpenAI adapter + testowy fallback PIL + retry
import base64
from io import BytesIO


def normalize_image_9x16(image, output_path):
    """Centralny crop do dokładnego 9:16, RGB i 1080×1920."""
    image = image.convert('RGB')
    target_ratio = 9 / 16
    w, h = image.size
    ratio = w / h
    if ratio > target_ratio:
        new_w = int(h * target_ratio)
        left = (w - new_w) // 2
        image = image.crop((left, 0, left + new_w, h))
    elif ratio < target_ratio:
        new_h = int(w / target_ratio)
        top = (h - new_h) // 2
        image = image.crop((0, top, w, top + new_h))
    image = image.resize((VIDEO_WIDTH, VIDEO_HEIGHT), Image.Resampling.LANCZOS)
    image.save(output_path, format='PNG', optimize=True)
    return Path(output_path)


def create_test_frame(scene, index, output_path):
    """Własny, syntetyczny kadr testowy — nie udaje wyniku modelu generującego obrazy."""
    seed = int(hashlib.sha256((scene['scene_id'] + scene['visual_prompt']).encode('utf-8')).hexdigest()[:8], 16)
    rng = random.Random(seed)
    width, height = 540, 960
    base = Image.new('RGB', (width, height))
    px = base.load()
    for y in range(height):
        t = y / max(1, height - 1)
        shade = int(22 + 34 * (1 - t))
        for x in range(width):
            noise = rng.randint(-7, 7)
            px[x, y] = (max(0, shade + noise - 5), max(0, shade + noise - 3), max(0, shade + noise))
    # Mgliste smugi i ziarnistość analogowa.
    fog = Image.new('RGBA', (width, height), (0, 0, 0, 0))
    fd = ImageDraw.Draw(fog)
    for _ in range(16):
        x = rng.randint(-120, width + 80)
        y = rng.randint(80, height - 100)
        rx, ry = rng.randint(45, 150), rng.randint(18, 65)
        fd.ellipse((x-rx, y-ry, x+rx, y+ry), fill=(175, 180, 180, rng.randint(8, 22)))
    base = Image.alpha_composite(base.convert('RGBA'), fog).convert('RGB')
    draw = ImageDraw.Draw(base)
    # Zarys opuszczonej zabudowy oraz przewodów antenowych; bez napisów.
    horizon = int(height * (0.58 + 0.04 * (index % 3)))
    for b in range(7):
        bw = rng.randint(38, 95)
        bx = b * 85 - 15 + rng.randint(-14, 14)
        bh = rng.randint(100, 270)
        color = (12 + rng.randint(0, 10), 14 + rng.randint(0, 10), 17 + rng.randint(0, 10))
        draw.rectangle((bx, horizon-bh, bx+bw, horizon+45), fill=color)
        for wy in range(horizon-bh+20, horizon, 34):
            for wx in range(bx+10, bx+bw-8, 25):
                if rng.random() > 0.78:
                    draw.rectangle((wx, wy, wx+7, wy+12), fill=(78, 74, 57))
    # Antena i pojedyncze światło w oddali.
    ax = width // 2 + rng.randint(-75, 75)
    draw.line((ax, horizon-260, ax, horizon-410), fill=(8, 9, 10), width=4)
    draw.line((ax-45, horizon-390, ax+45, horizon-390), fill=(8, 9, 10), width=3)
    glow = Image.new('RGBA', (width, height), (0, 0, 0, 0))
    gd = ImageDraw.Draw(glow)
    lx, ly = rng.randint(90, width-90), rng.randint(230, 560)
    gd.ellipse((lx-40, ly-40, lx+40, ly+40), fill=(194, 157, 89, 24))
    gd.ellipse((lx-8, ly-8, lx+8, ly+8), fill=(210, 181, 125, 105))
    base = Image.alpha_composite(base.convert('RGBA'), glow).convert('RGB')
    base = ImageEnhance.Contrast(base).enhance(1.15).filter(ImageFilter.GaussianBlur(0.35))
    grain = Image.effect_noise((width, height), 34).convert('RGB')
    base = Image.blend(base, grain, 0.10)
    return normalize_image_9x16(base, output_path)


def generate_openai_image(scene, output_path):
    if not OPENAI_API_KEY:
        raise RuntimeError('IMAGE_PROVIDER=openai wymaga OPENAI_API_KEY.')
    prompt = (
        f"{scene['visual_prompt']}. Style: {VISUAL_STYLE}. "
        "Vertical portrait composition, 9:16 framing, cinematic still frame, realistic lighting. "
        "No text, no letters, no logos, no subtitles, no watermark, no graphic gore."
    )
    response = requests.post(
        f'{OPENAI_BASE_URL}/images/generations',
        headers={'Authorization': f'Bearer {OPENAI_API_KEY}', 'Content-Type': 'application/json'},
        json={
            'model': OPENAI_IMAGE_MODEL,
            'prompt': prompt,
            'size': '1008x1792',
            'quality': 'medium',
            'output_format': 'png',
        },
        timeout=180,
    )
    response.raise_for_status()
    payload = response.json()
    entries = payload.get('data') or []
    if not entries:
        raise ValueError('API obrazów zwróciło pustą tablicę data.')
    item = entries[0]
    if item.get('b64_json'):
        image_bytes = base64.b64decode(item['b64_json'])
    elif item.get('url'):
        image_response = requests.get(item['url'], timeout=120)
        image_response.raise_for_status()
        image_bytes = image_response.content
    else:
        raise ValueError('Brak b64_json i url w odpowiedzi API obrazów.')
    image = Image.open(BytesIO(image_bytes))
    normalize_image_9x16(image, output_path)


def generate_frame_with_retry(scene, index):
    output_path = FRAME_EPISODE_DIR / f"{index:02d}_{scene['scene_id']}.png"
    last_error = None
    for attempt in range(1, IMAGE_RETRIES + 1):
        try:
            if IMAGE_PROVIDER == 'fallback':
                create_test_frame(scene, index, output_path)
            elif IMAGE_PROVIDER == 'openai':
                generate_openai_image(scene, output_path)
            elif IMAGE_PROVIDER in {'gemini', 'huggingface', 'together'}:
                raise NotImplementedError(
                    f"Adapter {IMAGE_PROVIDER} nie jest aktywowany. Wstaw wywołanie dostawcy w generate_frame_with_retry()."
                )
            else:
                raise ValueError("IMAGE_PROVIDER musi mieć wartość 'fallback' albo 'openai'.")
            if not output_path.exists() or output_path.stat().st_size < 1000:
                raise RuntimeError('Wygenerowany kadr jest pusty lub zbyt mały.')
            with Image.open(output_path) as check_image:
                if check_image.size != (VIDEO_WIDTH, VIDEO_HEIGHT):
                    raise ValueError(f'Nieprawidłowy rozmiar kadru: {check_image.size}')
            return output_path
        except Exception as exc:
            last_error = exc
            print(f"[OSTRZEŻENIE] Kadr {scene['scene_id']}, próba {attempt}/{IMAGE_RETRIES}: {type(exc).__name__}: {exc}")
            time.sleep(min(2 ** (attempt - 1), 8))
    if FALLBACK_ON_IMAGE_ERROR and IMAGE_PROVIDER != 'fallback':
        print(f"[INFO] Używam kadru testowego dla {scene['scene_id']}.")
        return create_test_frame(scene, index, output_path)
    raise RuntimeError(f"Nie udało się wygenerować kadru {scene['scene_id']}: {last_error}")


FRAME_PATHS = []
for idx, scene in enumerate(STORY['scenes'], start=1):
    FRAME_PATHS.append(generate_frame_with_retry(scene, idx))
    print(f'[OK] {scene["scene_id"]}: {FRAME_PATHS[-1].name}')

frame_manifest = {
    'episode_id': EPISODE_ID,
    'provider': IMAGE_PROVIDER,
    'frames': [str(path) for path in FRAME_PATHS],
}
(FRAME_EPISODE_DIR / 'manifest.json').write_text(json.dumps(frame_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'[OK] Gotowe kadry: {len(FRAME_PATHS)} | katalog: {FRAME_EPISODE_DIR}')

In [ ]:
# KOMÓRKA 6/6 — SRT, płynny zoom, ffmpeg, H.264 1080×1920 / 30 fps
from IPython.display import display, Video


def srt_timestamp(seconds):
    millis = max(0, int(round(seconds * 1000)))
    hours, rem = divmod(millis, 3_600_000)
    minutes, rem = divmod(rem, 60_000)
    secs, ms = divmod(rem, 1000)
    return f'{hours:02d}:{minutes:02d}:{secs:02d},{ms:03d}'


def wrap_caption(text, width=30):
    return '\n'.join(textwrap.wrap(text.strip(), width=width, break_long_words=False, break_on_hyphens=False))

# Hook + narracje scen mają osobne timingi z faktycznych plików TTS.
subtitle_entries = []
cursor = 0.0
for item, duration in zip(SEGMENT_SPECS, SEGMENT_DURATIONS):
    start, end = cursor, cursor + duration
    subtitle_entries.append((start, end, item['text']))
    cursor = end

srt_path = TEMP_DIR / f'{EPISODE_ID}.srt'
srt_lines = []
for number, (start, end, caption) in enumerate(subtitle_entries, start=1):
    srt_lines.extend([str(number), f'{srt_timestamp(start)} --> {srt_timestamp(end)}', wrap_caption(caption, width=30), ''])
srt_path.write_text('\n'.join(srt_lines), encoding='utf-8')

# Pierwszy obraz obejmuje hook + narrację sceny 1; kolejne obrazy odpowiadają kolejnym scenom.
clip_durations = [SEGMENT_DURATIONS[0] + SEGMENT_DURATIONS[1]] + SEGMENT_DURATIONS[2:]
if len(FRAME_PATHS) != len(clip_durations):
    raise RuntimeError(f'Liczba kadrów ({len(FRAME_PATHS)}) nie pasuje do liczby klipów ({len(clip_durations)}).')

clip_paths = []
for idx, (frame_path, duration) in enumerate(zip(FRAME_PATHS, clip_durations), start=1):
    clip_path = TEMP_DIR / f'{EPISODE_ID}_clip_{idx:02d}.mp4'
    frame_count = max(1, int(math.ceil(duration * FPS)))
    zoom_filter = (
        "zoompan=z='min(1.0+on/1800\\,1.08)':"
        "x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':"
        f"d=1:s={VIDEO_WIDTH}x{VIDEO_HEIGHT}:fps={FPS},setsar=1,format=yuv420p"
    )
    try:
        run_cmd([
            'ffmpeg', '-y', '-loop', '1', '-framerate', str(FPS), '-i', str(frame_path),
            '-vf', zoom_filter, '-frames:v', str(frame_count), '-an',
            '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '20', '-pix_fmt', 'yuv420p',
            '-r', str(FPS), str(clip_path)
        ])
    except Exception as exc:
        raise RuntimeError(f'Nie udało się wyrenderować ujęcia {idx}: {exc}') from exc
    clip_paths.append(clip_path)
    print(f'[OK] Klip {idx}/{len(FRAME_PATHS)}: {duration:.2f} s')

# Lista concat bezpieczna dla ścieżek w Google Drive.
concat_video_list = TEMP_DIR / f'{EPISODE_ID}_video_concat.txt'
concat_video_list.write_text('\n'.join(f"file '{p.as_posix()}'" for p in clip_paths), encoding='utf-8')
SILENT_VIDEO = TEMP_DIR / f'{EPISODE_ID}_silent_video.mp4'
run_cmd([
    'ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', str(concat_video_list),
    '-an', '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '19',
    '-pix_fmt', 'yuv420p', '-r', str(FPS), str(SILENT_VIDEO)
])

OUTPUT_VIDEO = OUTPUT_DIR / f'{EPISODE_ID}.mp4'
subtitle_filter = (
    f"subtitles=filename='{srt_path.as_posix()}':"
    "force_style='FontName=DejaVu Sans,FontSize=44,PrimaryColour=&H00FFFFFF,"
    "OutlineColour=&HCC000000,BorderStyle=1,Outline=3,Shadow=1,Alignment=2,"
    "MarginV=320,MarginL=100,MarginR=100'"
)
try:
    run_cmd([
        'ffmpeg', '-y', '-i', str(SILENT_VIDEO), '-i', str(FINAL_AUDIO),
        '-vf', subtitle_filter, '-map', '0:v:0', '-map', '1:a:0', '-shortest',
        '-c:v', 'libx264', '-preset', 'medium', '-crf', '20', '-pix_fmt', 'yuv420p',
        '-r', str(FPS), '-c:a', 'aac', '-b:a', '192k', '-movflags', '+faststart',
        str(OUTPUT_VIDEO)
    ])
except Exception as exc:
    raise RuntimeError(
        'Eksport MP4 nie powiódł się. Sprawdź, czy ffmpeg zawiera filtr subtitles/libass, '
        f'czy plik SRT istnieje i czy ścieżki nie zawierają nietypowych znaków. Szczegóły: {exc}'
    ) from exc

# Walidacja końcowego pliku.
if not OUTPUT_VIDEO.exists() or OUTPUT_VIDEO.stat().st_size < 100_000:
    raise RuntimeError(f'Plik wyjściowy nie istnieje lub jest podejrzanie mały: {OUTPUT_VIDEO}')
probe = run_cmd(['ffprobe', '-v', 'error', '-show_streams', '-show_format', '-of', 'json', str(OUTPUT_VIDEO)])
probe_data = json.loads(probe.stdout)
video_stream = next((s for s in probe_data.get('streams', []) if s.get('codec_type') == 'video'), None)
audio_stream = next((s for s in probe_data.get('streams', []) if s.get('codec_type') == 'audio'), None)
if not video_stream or int(video_stream.get('width', 0)) != VIDEO_WIDTH or int(video_stream.get('height', 0)) != VIDEO_HEIGHT:
    raise RuntimeError('Walidacja rozdzielczości nie powiodła się — oczekiwano 1080×1920.')
if video_stream.get('r_frame_rate') != f'{FPS}/1':
    raise RuntimeError(f"Nieoczekiwana liczba klatek: {video_stream.get('r_frame_rate')} (oczekiwano {FPS}/1).")
if video_stream.get('codec_name') != 'h264' or not audio_stream or audio_stream.get('codec_name') != 'aac':
    raise RuntimeError('Walidacja kodeka nie powiodła się — oczekiwano H.264, AAC i obecności audio.')

final_duration = float(probe_data.get('format', {}).get('duration', 0))
print('\n' + '=' * 60)
print('[GOTOWE] Lost-words / Zgubione słowa')
print('Tytuł:', STORY['title'])
print('Plik:', OUTPUT_VIDEO)
print(f'Parametry: {VIDEO_WIDTH}×{VIDEO_HEIGHT}, {FPS} fps, H.264, AAC, {final_duration:.2f} s')
print('Tryb audio:', 'cisza testowa' if all('silent' in p.name for p in AUDIO_SEGMENT_FILES) else 'edge-tts')
print('Tryb kadrów:', IMAGE_PROVIDER)
print('=' * 60)

display(Video(str(OUTPUT_VIDEO), embed=True, width=360))